In [1]:
import ee
import geemap
from dotenv import load_dotenv
import os
import ipywidgets as widgets
from datetime import datetime
import matplotlib.pyplot as plt
import pandas as pd

load_dotenv(dotenv_path='.env')
project_name = os.getenv("project_name")
# print(f"Project Name: {project_name}")
ee.Initialize(project=project_name)

In [2]:
map = geemap.Map()

point = ee.Geometry.Point([90.4152, 23.8041]) #around dhaka
region = ee.Geometry.Rectangle([90.3, 23.7, 90.5, 23.9]) #bounding box around the point

top_left = [23.822424724001266, 90.46289150228976]
bottom_right = [23.796369273111445, 90.5071668854189]

#region = ee.Geometry.Rectangle([top_left[1], bottom_right[0], bottom_right[1], top_left[0]])
region = ee.Geometry.Rectangle([92.12743533806102, 21.81144778755447, 92.56414188102977, 22.207414604160263])

map.centerObject(region, 10)
map.add_basemap('HYBRID')
map.addLayer(point, {'color': 'red'}, 'Point Layer')
map.addLayer(region, {'color': 'blue'}, 'Region Layer')


map.default_style = {'cursor': 'crosshair'}


map #just testing if everything is working

Map(center=[22.009483577149734, 92.34578860954535], controls=(WidgetControl(options=['position', 'transparent_…

In [ ]:
#now loading the elevation data - NASADEM

dem = ee.Image("NASA/NASADEM_HGT/001").select('elevation')

elevation = dem.clip(region)

slope = ee.Terrain.slope(elevation)

floodRisk = elevation.lt(10).And(slope.lt(5)) #it just scans the area where elevation is less than 10 meters and slope is less than 5 degrees

landslideRisk = slope.gt(20) #scans area where slope is greater than 20 degrees

In [ ]:
m1 = geemap.Map()
m1.centerObject(region, 10)


stats_label = widgets.Label(value='Calculating area...')
slider = widgets.IntSlider(
    value=20,
    min=0,
    max=90,
    step=1,
    description='Slope (>):',
    continuous_update=False  
)
def update_map(change):
    threshold = change['new']
    risk_mask = slope.gt(threshold)
    
    #remove previous layer if exists and add new layer
    if m1.find_layer('dynamic risk zone'):
        m1.remove_layer('dynamic risk zone')
        
    m1.addLayer(risk_mask.selfMask(), {'palette': 'red'}, 'dynamic risk zone')
    
    #calculation stastics
    stats_label.value = 'calculating'
    
    
    try:
        risk_stats = risk_mask.reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=region,
            scale=30,
            maxPixels=1e9
        )
        
        #extract pixel count
        pixel_count = risk_stats.get('slope').getInfo()
        
        if pixel_count is None:
            pixel_count = 0
            
        area_sq_km = (pixel_count * 900) / 1e6
        stats_label.value = f'area> {threshold}° is {area_sq_km:.2f} km²'
        
    except Exception as e:
        stats_label.value = f'error: {e}'

slider.observe(update_map, names='value')
panel = widgets.VBox([
    widgets.Label('Slope Risk Analyzer', style={'font_weight': 'bold', 'font_size': '18px'}),
    slider,
    stats_label
])
m1.add_widget(panel)
m1



Map(center=[22.009483577149734, 92.34578860954535], controls=(WidgetControl(options=['position', 'transparent_…